# 06 — Interactive H₂ Yield Prediction

Run **Cell 1**. Popup windows ask for **10 primary inputs** (with units).
Derived variables and the predicted H₂ yield are shown **with units**.

## Variable units (from original dataset, Notebook 01)

| Variable | Unit |
|---|---|
| Substrate / inoculum MC, TS, VS, FS | % |
| SCOD, TCOD, TRS, HMF (5-HMF) | mg/L |
| Lignin | % |
| SCOD/TCOD ratio | dimensionless |
| F/M ratio | dimensionless |
| **H₂ yield (output)** | **mL H₂/g VS** |

## Derived automatically

| You enter | Derived |
|---|---|
| Substrate MC (%) | Substrate TS (%) = 100 − MC |
| Substrate VS (%) | Substrate FS (%) = 100 − VS |
| Inoculum MC (%) | Inoculum TS (%) = 100 − MC |
| Inoculum VS (%) | Inoculum FS (%) = 100 − VS |
| SCOD, TCOD (mg/L) | SCOD/TCOD ratio = SCOD ÷ TCOD |

**Prerequisite:** Notebook 05 Section 38 saved the model file.

In [7]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import pandas as pd
from pathlib import Path
import tkinter as tk
from tkinter import simpledialog

H2_YIELD_UNIT = "mL H₂/g VS"

PRIMARY_LABELS = {
    "substrate_mc": "Substrate moisture content (MC)",
    "substrate_vs": "Substrate volatile solids (VS)",
    "inoculum_mc": "Inoculum moisture content (MC)",
    "inoculum_vs": "Inoculum volatile solids (VS)",
    "scod": "SCOD",
    "tcod": "TCOD",
    "trs": "TRS",
    "lignin": "Lignin",
    "hmf": "HMF (5-HMF)",
    "fm_ratio": "Food-to-microorganism (F/M) ratio",
}

UNITS = {
    "substrate_mc": "%",
    "substrate_vs": "%",
    "inoculum_mc": "%",
    "inoculum_vs": "%",
    "scod": "mg/L",
    "tcod": "mg/L",
    "trs": "mg/L",
    "lignin": "%",
    "hmf": "mg/L",
    "fm_ratio": "dimensionless",
    "substrate_ts": "%",
    "substrate_fs": "%",
    "inoculum_ts": "%",
    "inoculum_fs": "%",
    "scod_tcod_ratio": "dimensionless",
    "h2_yield": H2_YIELD_UNIT,
}

PRIMARY_DEFAULTS = {
    "substrate_mc": 5.26,
    "substrate_vs": 89.41,
    "inoculum_mc": 89.92,
    "inoculum_vs": 90.30,
    "scod": 6000.0,
    "tcod": 11400.0,
    "trs": 1.0,
    "lignin": 25.0,
    "hmf": 45.0,
    "fm_ratio": 1.0,
}

PRIMARY_ORDER = list(PRIMARY_LABELS.keys())
VALID_FM_RATIOS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]


def format_value(value, unit, decimals=2):
    if unit == "dimensionless":
        return f"{value:.{decimals}f} (dimensionless)"
    return f"{value:.{decimals}f} {unit}"


def label_with_unit(field):
    unit = UNITS[field]
    if unit == "dimensionless":
        return f"{PRIMARY_LABELS[field]} (dimensionless)"
    return f"{PRIMARY_LABELS[field]} ({unit})"


def derive_model_features(primary):
    if primary["tcod"] == 0:
        raise ValueError("TCOD must be greater than zero.")

    return {
        "substrate_mc": primary["substrate_mc"],
        "substrate_ts": 100.0 - primary["substrate_mc"],
        "substrate_vs": primary["substrate_vs"],
        "substrate_fs": 100.0 - primary["substrate_vs"],
        "inoculum_mc": primary["inoculum_mc"],
        "inoculum_ts": 100.0 - primary["inoculum_mc"],
        "inoculum_vs": primary["inoculum_vs"],
        "inoculum_fs": 100.0 - primary["inoculum_vs"],
        "scod": primary["scod"],
        "tcod": primary["tcod"],
        "scod_tcod_ratio": primary["scod"] / primary["tcod"],
        "trs": primary["trs"],
        "lignin": primary["lignin"],
        "hmf": primary["hmf"],
        "fm_ratio": primary["fm_ratio"],
    }


def print_derived_values(features):
    print("\nDerived values:")
    print(
        "  Substrate TS        :",
        format_value(features["substrate_ts"], UNITS["substrate_ts"]),
    )
    print(
        "  Substrate FS        :",
        format_value(features["substrate_fs"], UNITS["substrate_fs"]),
    )
    print(
        "  Inoculum TS         :",
        format_value(features["inoculum_ts"], UNITS["inoculum_ts"]),
    )
    print(
        "  Inoculum FS         :",
        format_value(features["inoculum_fs"], UNITS["inoculum_fs"]),
    )
    print(
        "  SCOD/TCOD ratio     :",
        format_value(
            features["scod_tcod_ratio"],
            UNITS["scod_tcod_ratio"],
            decimals=4,
        ),
    )


def find_pipeline_path():
    candidates = [
        Path("models/h2_yield_pipeline.joblib"),
        Path("data/models/h2_yield_pipeline.joblib"),
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(
        "Model not found. Expected one of:\n"
        "  models/h2_yield_pipeline.joblib\n"
        "  data/models/h2_yield_pipeline.joblib\n"
        "Run Notebook 05 Section 38 or data/train_and_save_pipeline.py first."
    )


PIPELINE_PATH = find_pipeline_path()

pipeline = joblib.load(PIPELINE_PATH)
model = pipeline["model"]
feature_columns = pipeline["feature_columns"]

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

print("10 popup windows will open for primary inputs.")
print("TS, FS, and SCOD/TCOD ratio are calculated automatically.\n")

primary_input = {}

for i, field in enumerate(PRIMARY_ORDER, start=1):
    label = label_with_unit(field)
    default = PRIMARY_DEFAULTS[field]
    unit = UNITS[field]

    value = simpledialog.askfloat(
        title=f"Input {i} of {len(PRIMARY_ORDER)}",
        prompt=(
            f"{label}\n\n"
            f"Default: {format_value(default, unit)}\n"
            f"Cancel = use default"
        ),
        initialvalue=default,
        parent=root,
    )

    primary_input[field] = default if value is None else value
    print(
        f"  {label}: "
        f"{format_value(primary_input[field], unit)}"
    )

root.destroy()

if primary_input["fm_ratio"] not in VALID_FM_RATIOS:
    print(
        f"\nWarning: F/M ratio {primary_input['fm_ratio']} is outside "
        f"{VALID_FM_RATIOS}."
    )

model_features = derive_model_features(primary_input)
print_derived_values(model_features)

X_new = pd.DataFrame([model_features])[feature_columns]
predicted_h2 = float(model.predict(X_new)[0])

print("\n" + "=" * 60)
print("PREDICTION RESULT")
print("=" * 60)
print(f"Model              : {pipeline['model_name']}")
print(
    f"F/M ratio          : "
    f"{format_value(primary_input['fm_ratio'], UNITS['fm_ratio'])}"
)
print(
    f"Predicted H₂ yield : "
    f"{format_value(predicted_h2, H2_YIELD_UNIT)}"
)
print("=" * 60)

10 popup windows will open for primary inputs.
TS, FS, and SCOD/TCOD ratio are calculated automatically.

  Substrate moisture content (MC) (%): 6.15 %
  Substrate volatile solids (VS) (%): 91.22 %
  Inoculum moisture content (MC) (%): 90.25 %
  Inoculum volatile solids (VS) (%): 92.45 %
  SCOD (mg/L): 6400.00 mg/L
  TCOD (mg/L): 10000.00 mg/L
  TRS (mg/L): 256.33 mg/L
  Lignin (%): 16.32 %
  HMF (5-HMF) (mg/L): 316.16 mg/L
  Food-to-microorganism (F/M) ratio (dimensionless): 2.00 (dimensionless)

Derived values:
  Substrate TS        : 93.85 %
  Substrate FS        : 8.78 %
  Inoculum TS         : 9.75 %
  Inoculum FS         : 7.55 %
  SCOD/TCOD ratio     : 0.6400 (dimensionless)

PREDICTION RESULT
Model              : Tuned Extra Trees
F/M ratio          : 2.00 (dimensionless)
Predicted H₂ yield : 278.90 mL H₂/g VS


## Fallback — text prompts (same 10 primary inputs, with units)

Run if popups do not appear. Type answers in the **input box at the top**
of the notebook when the cell is running (`In [*]`).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import pandas as pd
from pathlib import Path

H2_YIELD_UNIT = "mL H₂/g VS"

PRIMARY_LABELS = {
    "substrate_mc": "Substrate moisture content (MC)",
    "substrate_vs": "Substrate volatile solids (VS)",
    "inoculum_mc": "Inoculum moisture content (MC)",
    "inoculum_vs": "Inoculum volatile solids (VS)",
    "scod": "SCOD",
    "tcod": "TCOD",
    "trs": "TRS",
    "lignin": "Lignin",
    "hmf": "HMF (5-HMF)",
    "fm_ratio": "Food-to-microorganism (F/M) ratio",
}

UNITS = {
    "substrate_mc": "%",
    "substrate_vs": "%",
    "inoculum_mc": "%",
    "inoculum_vs": "%",
    "scod": "mg/L",
    "tcod": "mg/L",
    "trs": "mg/L",
    "lignin": "%",
    "hmf": "mg/L",
    "fm_ratio": "dimensionless",
    "substrate_ts": "%",
    "substrate_fs": "%",
    "inoculum_ts": "%",
    "inoculum_fs": "%",
    "scod_tcod_ratio": "dimensionless",
}

PRIMARY_DEFAULTS = {
    "substrate_mc": 5.26,
    "substrate_vs": 89.41,
    "inoculum_mc": 89.92,
    "inoculum_vs": 90.30,
    "scod": 6000.0,
    "tcod": 11400.0,
    "trs": 1.0,
    "lignin": 25.0,
    "hmf": 45.0,
    "fm_ratio": 1.0,
}

PRIMARY_ORDER = list(PRIMARY_LABELS.keys())


def format_value(value, unit, decimals=2):
    if unit == "dimensionless":
        return f"{value:.{decimals}f} (dimensionless)"
    return f"{value:.{decimals}f} {unit}"


def label_with_unit(field):
    unit = UNITS[field]
    if unit == "dimensionless":
        return f"{PRIMARY_LABELS[field]} (dimensionless)"
    return f"{PRIMARY_LABELS[field]} ({unit})"


def derive_model_features(primary):
    if primary["tcod"] == 0:
        raise ValueError("TCOD must be greater than zero.")

    return {
        "substrate_mc": primary["substrate_mc"],
        "substrate_ts": 100.0 - primary["substrate_mc"],
        "substrate_vs": primary["substrate_vs"],
        "substrate_fs": 100.0 - primary["substrate_vs"],
        "inoculum_mc": primary["inoculum_mc"],
        "inoculum_ts": 100.0 - primary["inoculum_mc"],
        "inoculum_vs": primary["inoculum_vs"],
        "inoculum_fs": 100.0 - primary["inoculum_vs"],
        "scod": primary["scod"],
        "tcod": primary["tcod"],
        "scod_tcod_ratio": primary["scod"] / primary["tcod"],
        "trs": primary["trs"],
        "lignin": primary["lignin"],
        "hmf": primary["hmf"],
        "fm_ratio": primary["fm_ratio"],
    }


def find_pipeline_path():
    candidates = [
        Path("models/h2_yield_pipeline.joblib"),
        Path("data/models/h2_yield_pipeline.joblib"),
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(
        "Model not found. Run Notebook 05 Section 38 first."
    )


PIPELINE_PATH = find_pipeline_path()
pipeline = joblib.load(PIPELINE_PATH)
model = pipeline["model"]
feature_columns = pipeline["feature_columns"]

print("=" * 60)
print("10 PRIMARY INPUTS — TS/FS/ratio will be derived")
print("Use the input box at the TOP of the notebook.")
print("=" * 60 + "\n")

primary_input = {}

for i, field in enumerate(PRIMARY_ORDER, start=1):
    label = label_with_unit(field)
    default = PRIMARY_DEFAULTS[field]
    unit = UNITS[field]

    while True:
        answer = input(
            f"({i}/{len(PRIMARY_ORDER)}) {label} "
            f"[{format_value(default, unit)}]: "
        ).strip()

        if answer == "":
            value = default
            break
        try:
            value = float(answer)
            break
        except ValueError:
            print("  Please enter a valid number.")

    primary_input[field] = value

model_features = derive_model_features(primary_input)

print("\nDerived values:")
print(
    "  Substrate TS    :",
    format_value(model_features["substrate_ts"], UNITS["substrate_ts"]),
)
print(
    "  Substrate FS    :",
    format_value(model_features["substrate_fs"], UNITS["substrate_fs"]),
)
print(
    "  Inoculum TS     :",
    format_value(model_features["inoculum_ts"], UNITS["inoculum_ts"]),
)
print(
    "  Inoculum FS     :",
    format_value(model_features["inoculum_fs"], UNITS["inoculum_fs"]),
)
print(
    "  SCOD/TCOD ratio :",
    format_value(
        model_features["scod_tcod_ratio"],
        UNITS["scod_tcod_ratio"],
        decimals=4,
    ),
)

X_new = pd.DataFrame([model_features])[feature_columns]
predicted_h2 = float(model.predict(X_new)[0])

print("\n" + "=" * 60)
print(
    f"Predicted H₂ yield : "
    f"{format_value(predicted_h2, H2_YIELD_UNIT)}"
)
print("=" * 60)